# 실습 9: Ollama LLM을 활용한 HTTP 분류 (2교시)

특성 추출 없이 **HTTP 요청 텍스트를 그대로** Ollama gemma3:4b에 보여주고
정상/공격을 분류합니다.

**사전 조건**:
- Ollama 서버 실행 중 (`ollama serve`)
- `gemma3:4b` 모델 다운로드됨 (`ollama list`로 확인)
- 7주차 `processed_data.pkl` 존재 (LLM용 텍스트 샘플 포함)


In [1]:
# %% [Setup] 패키지 import 및 LLM용 샘플 로드
import pickle
import time
import json
import re
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from urllib.parse import unquote
from sklearn.metrics import accuracy_score, f1_score, classification_report

import ollama  # pip install ollama

with open("../7주차/processed_data.pkl", "rb") as f:
    data = pickle.load(f)

llm_sample = data["llm_sample"].head(100).reset_index(drop=True)
print(f"분류 대상: {len(llm_sample)}건")
print(f"라벨 분포: 정상 {(llm_sample.get('is_attack',0)==0).sum()}건 / "
      f"공격 {(llm_sample.get('is_attack',0)==1).sum()}건")


분류 대상: 100건
라벨 분포: 정상 56건 / 공격 44건


## 1. 분류 프롬프트 설계

LLM 응답을 안정적으로 파싱하기 위해:
- **Few-shot 예시** 2개로 출력 형식을 학습시킴
- **JSON 형태**로 응답하도록 강제
- 영어 프롬프트(gemma3:4b가 영어에 더 정확)

In [11]:
# %% [1] HTTP 텍스트 재구성 + 프롬프트 함수
def build_http_text(row) -> str:
    method = row.get("method", "GET")
    url    = unquote(str(row.get("url", "")), encoding="latin-1")
    body   = str(row.get("body_decoded", row.get("body", "")) or "")
    text   = f"{method} {url} HTTP/1.1"
    if body and body != "nan":
        text += f"\nBody: {body[:200]}"
    return text


PROMPT_TEMPLATE = 'You are a web security expert. Classify each HTTP request as "Normal" or "Anomalous" and provide a brief reason.\n\nExamples:\nRequest: GET /index.jsp HTTP/1.1\nOutput: {{"label": "Normal", "reason": "Standard page request, no suspicious pattern"}}\n\nRequest: GET /search?q=\' OR \'1\'=\'1 HTTP/1.1\nOutput: {{"label": "Anomalous", "reason": "Classic SQL Injection pattern with OR 1=1"}}\n\nNow classify:\nRequest: {http_text}\nOutput:'


def classify_with_llm(http_text: str, model: str = "gemma3:4b") -> dict:
    """Ollama로 HTTP 요청 분류 -> {label, reason}"""
    prompt = PROMPT_TEMPLATE.format(http_text=http_text)
    response = ollama.chat(
        model=model,
        messages=[{"role":"user","content":prompt}],
        options={"temperature": 0},  # 결정성 높이기
    )
    text = response["message"]["content"]

    # JSON 추출 - LLM이 가끔 앞뒤 설명을 붙임
    match = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if not match:
        return {"label":"Unknown", "reason": text[:80]}
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return {"label":"Unknown", "reason": text[:80]}


# 단건 테스트
test_text = "GET /tienda1/publico/anadir.jsp?id=2'+OR+'1'='1 HTTP/1.1"
print("입력:", test_text)
print("응답:", classify_with_llm(test_text))


입력: GET /tienda1/publico/anadir.jsp?id=2'+OR+'1'='1 HTTP/1.1
응답: {'label': 'Anomalous', 'reason': "SQL Injection attempt. The request includes ' OR '1'='1', a common SQL injection pattern to bypass authentication or retrieve all data."}


## 2. 100건 분류 + 시간 측정

CPU 환경 기준 건당 1~3초가 표준. 100건 ≈ 2~5분 소요.

In [12]:
# %% [2] 100건 분류
results = []
start = time.time()

for i, row in llm_sample.iterrows():
    http_text = build_http_text(row)
    result = classify_with_llm(http_text)
    true_label = "Anomalous" if row.get("is_attack", 0) == 1 else "Normal"
    results.append({
        "idx": i,
        "true": true_label,
        "pred": result.get("label", "Unknown"),
        "reason": result.get("reason", "")[:120],
        "http_short": http_text[:100],
    })
    if (i + 1) % 10 == 0:
        elapsed = time.time() - start
        print(f"  {i+1}/{len(llm_sample)}건 완료 "
              f"({elapsed:.1f}초, 건당 {elapsed/(i+1):.2f}초)")

llm_time = time.time() - start
llm_df = pd.DataFrame(results)
print(f"\n총 소요: {llm_time:.1f}초")
print(f"1만 건 환산: 약 {llm_time/100*10000/60:.0f}분")


  10/100건 완료 (8.1초, 건당 0.81초)
  20/100건 완료 (14.3초, 건당 0.71초)
  30/100건 완료 (21.0초, 건당 0.70초)
  40/100건 완료 (28.9초, 건당 0.72초)
  50/100건 완료 (37.0초, 건당 0.74초)
  60/100건 완료 (45.3초, 건당 0.76초)
  70/100건 완료 (51.6초, 건당 0.74초)
  80/100건 완료 (59.7초, 건당 0.75초)
  90/100건 완료 (67.1초, 건당 0.75초)
  100/100건 완료 (73.9초, 건당 0.74초)

총 소요: 73.9초
1만 건 환산: 약 123분


In [13]:
# %% [3] 정확도/F1 계산
llm_df["pred_clean"] = llm_df["pred"].replace({"Unknown":"Normal"})
y_true = (llm_df["true"] == "Anomalous").astype(int)
y_pred = (llm_df["pred_clean"] == "Anomalous").astype(int)

llm_acc = accuracy_score(y_true, y_pred)
llm_f1  = f1_score(y_true, y_pred)

print(f"LLM 정확도: {llm_acc:.4f}")
print(f"LLM F1:    {llm_f1:.4f}")
print(f"분류 실패(Unknown): {(llm_df['pred']=='Unknown').sum()}건")
print()
print(classification_report(y_true, y_pred, target_names=["Normal","Anomalous"]))


LLM 정확도: 0.8300
LLM F1:    0.8211
분류 실패(Unknown): 1건

              precision    recall  f1-score   support

      Normal       0.90      0.79      0.84        56
   Anomalous       0.76      0.89      0.82        44

    accuracy                           0.83       100
   macro avg       0.83      0.84      0.83       100
weighted avg       0.84      0.83      0.83       100



## 3. 자연어 판단 근거 검토 ★

LLM의 가장 큰 강점: **왜 그렇게 판단했는지** 사람이 읽을 수 있는 문장으로 설명.
이는 SOC(보안관제) 분석가가 1차 분류를 검토할 때 매우 유용합니다.

In [14]:
# %% [4] 공격으로 판단한 사례 + LLM 근거
print("=== LLM이 공격으로 판단한 사례 (상위 5건) ===\n")
attack_pred = llm_df[llm_df["pred"] == "Anomalous"].head(5)
for _, r in attack_pred.iterrows():
    correct = "OK" if r["true"] == "Anomalous" else "오탐"
    print(f"[{correct}] 실제={r['true']:10s}  요청: {r['http_short']}")
    print(f"   - LLM 근거: {r['reason']}\n")


=== LLM이 공격으로 판단한 사례 (상위 5건) ===

[OK] 실제=Anomalous   요청: GET /tienda1/miembros/editar.jsp?modo=registro&loginA=lieure&password=rEbatible&nombre=Tarciano&apel
   - LLM 근거: The request contains numerous URL parameters, including unusual characters and potentially malicious-looking values like

[OK] 실제=Anomalous   요청: GET /admin/login.do HTTP/1.1
   - LLM 근거: Requests to '/admin/' endpoints are often anomalous and require careful scrutiny, particularly login pages.  This sugges

[오탐] 실제=Normal      요청: POST /tienda1/publico/entrar.jsp HTTP/1.1
Body: errorMsg=Credenciales+incorrectas
   - LLM 근거: POST request to a potentially sensitive endpoint (/tienda1/publico/entrar.jsp) with a body containing an error message s

[오탐] 실제=Normal      요청: GET /tienda1/publico/anadir.jsp?id=1&nombre=Jamón+Ibérico&precio=100&cantidad=22&B1=Añadir+al+carrit
   - LLM 근거: The request contains a parameter 'B1' which doesn't align with typical e-commerce flow (adding to cart). This could be a

[OK] 실제=Anomalous

In [9]:
# %% [5] 결과 저장 (3교시에서도 활용)
with open("llm_classification_results.pkl", "wb") as f:
    pickle.dump({
        "llm_df": llm_df,
        "llm_acc": llm_acc,
        "llm_f1": llm_f1,
        "llm_time": llm_time,
        "n_samples": len(llm_sample),
    }, f)
print(">> llm_classification_results.pkl 저장 완료")


>> llm_classification_results.pkl 저장 완료


**다음**: `comparison_analysis.ipynb`로 1교시 ML 결과와 종합 비교합니다.

In [15]:
pip install --upgrade scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import pickle
import time
import ollama
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore') # 버전 경고창 숨기기

# 1. 데이터 로드 및 샘플링 (이전 단계 코드 연동)
with open("../7주차/processed_data.pkl", "rb") as f:
    data = pickle.load(f)

# 빠른 테스트를 위해 상위 20개만 먼저 진행 (확인 후 100개로 늘리세요)
llm_sample = data["llm_sample"].head(20).reset_index(drop=True)

# 2. 비교할 프롬프트 정의 (System Prompt 변경)
prompts = {
    "Prompt_A (기본 지시)": """
    당신은 보안 전문가입니다. 제공되는 데이터가 웹 공격이면 '1', 정상이면 '0'을 반환하세요.
    다른 설명은 절대 하지 말고 오직 숫자 '1' 또는 '0' 하나만 출력하세요.
    """,
    
    "Prompt_B (역할 및 규칙 강화)": """
    [Role] Web Security Analyst Bot
    [Task] Analyze the log and classify malicious behavior.
    [Rule] 
    - If it's a cyber attack (SQLi, XSS, etc.), output '1'.
    - If it's normal traffic, output '0'.
    - Output ONLY the single digit ('1' or '0'). No reasoning, no markdown, no spaces.
    """
}

# Ollama 호출 함수 정의
def get_llm_response(system_prompt, user_content, model_name="llama3"):
    try:
        response = ollama.chat(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": str(user_content)}
            ],
            options={"temperature": 0.0} # 결과의 일관성을 위해 온도를 0으로 설정
        )
        # 공백 제거 후 첫 글자만 추출 (숫자만 남기기 위함)
        result = response['message']['content'].strip()
        return result
    except Exception as e:
        print(f"Error: {e}")
        return "None"

# 3. 프롬프트별 실험 진행
results_summary = {}

for p_name, sys_prompt in prompts.items():
    print(f"\n🚀 {p_name} 테스트 시작...")
    y_pred = []
    
    start_time = time.time()
    for idx, row in llm_sample.iterrows():
        # 데이터의 특정 텍스트나 로그 컬럼을 넘겨줍니다. (컬럼명에 맞게 수정 필요)
        # 여기서는 row 전체를 문자열로 넘기거나 특정 컬럼(예: 'payload')을 지정하세요.
        user_input = row.to_dict() 
        
        pred = get_llm_response(sys_prompt, user_input)
        
        # 안전장치: 숫자만 파싱하기 위한 전처리
        if '1' in pred:
            y_pred.append(1)
        elif '0' in pred:
            y_pred.append(0)
        else:
            y_pred.append(0) # 예외 발생 시 기본값 정상(0) 처리
            
        if (idx + 1) % 5 == 0:
            print(f" 진행 중... ({idx + 1}/{len(llm_sample)})")
            
    end_time = time.time()
    
    # 실제 정답 라벨 (is_attack 컬럼 가정)
    y_true = llm_sample['is_attack'].tolist()
    
    # 평가 지표 계산
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    
    # 결과 저장
    results_summary[p_name] = {
        "Accuracy": acc,
        "F1-Score": f1,
        "Time (s)": round(end_time - start_time, 2),
        "Predictions": y_pred
    }

# 4. 최종 정확도 비교 결과 출력
print("\n" + "="*50)
print("📊 프롬프트 성능 비교 최종 결과")
print("="*50)
df_results = pd.DataFrame(results_summary).T
print(df_results[["Accuracy", "F1-Score", "Time (s)"]])


🚀 Prompt_A (기본 지시) 테스트 시작...
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
 진행 중... (5/20)
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
 진행 중... (10/20)
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
 진행 중... (15/20)
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
Error: model 'llama3' not found (status code: 404)
Er

In [3]:
import pandas as pd
import pickle
import time
import ollama
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 로드
with open("../7주차/processed_data.pkl", "rb") as f:
    data = pickle.load(f)

# Few-shot 예시를 추출할 학습 데이터와 테스트할 샘플 데이터 분리
X_train = data["X_train"]  # 필요시 데이터프레임 형태로 변환하여 예시 추출에 사용
y_train = data["y_train"]
llm_sample = data["llm_sample"].head(20).reset_index(drop=True) # 테스트용 20건

# -------------------------------------------------------------
# [💡 핵심] Few-shot으로 사용할 예시 데이터 동적 생성
# 실제 데이터의 컬럼 상황에 맞게 텍스트 예시를 직접 적거나, 아래처럼 샘플링합니다.
# -------------------------------------------------------------
# 예시로 사용할 정상(0)과 공격(1) 데이터 텍스트를 임의로 정의하거나 추출합니다.
normal_example_1 = "{'method': 'GET', 'path': '/index.html', 'user_agent': 'Mozilla/5.0'}"
normal_example_2 = "{'method': 'POST', 'path': '/login', 'user_agent': 'Mozilla/5.0'}"

attack_example_1 = "{'method': 'GET', 'path': '/admin?id=1 OR 1=1', 'user_agent': 'sqlmap'}"
attack_example_2 = "{'method': 'GET', 'path': '/<script>alert(1)</script>', 'user_agent': 'Mozilla'}"

# 2. 비교 실험할 Few-shot 프롬프트 후보들 정의
few_shot_prompts = {
    "Few-Shot_기본 (정상1, 공격1)": f"""
    당신은 보안 분석가입니다. 제공된 데이터가 웹 공격이면 '1', 정상이면 '0'을 반환하세요.
    오직 '1' 또는 '0'만 출력해야 합니다.
    
    [Example 1]
    Input: {normal_example_1}
    Output: 0
    
    [Example 2]
    Input: {attack_example_1}
    Output: 1
    """,
    
    "Few-Shot_강화 (정상2, 공격2 + 설명 형식 지정)": f"""
    [Role] Web Security Specialist
    [Task] Classify the input data into Normal (0) or Attack (1).
    [Rule] Output ONLY '0' or '1'. Do not write anything else.
    
    [Example 1 - Normal]
    Input: {normal_example_1}
    Output: 0
    
    [Example 2 - Normal]
    Input: {normal_example_2}
    Output: 0
    
    [Example 3 - Attack]
    Input: {attack_example_1}
    Output: 1
    
    [Example 4 - Attack]
    Input: {attack_example_2}
    Output: 1
    """
}

# Ollama 호출 함수
def get_ollama_few_shot(system_prompt, user_content, model_name="llama3"):
    try:
        response = ollama.chat(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Input: {str(user_content)}\nOutput:"} # Few-shot 유도를 위해 형식을 맞춰줌
            ],
            options={"temperature": 0.0} # 실험 일관성을 위해 0으로 고정
        )
        result = response['message']['content'].strip()
        return result
    except Exception as e:
        return "None"

# 3. Few-shot 프롬프트별 성능 비교 실험 시작
few_shot_results = {}

for p_name, sys_prompt in few_shot_prompts.items():
    print(f"\n🚀 {p_name} 테스트 시작...")
    y_pred = []
    start_time = time.time()
    
    for idx, row in llm_sample.iterrows():
        user_input = row.to_dict() # 검사할 데이터
        
        pred = get_ollama_few_shot(sys_prompt, user_input)
        
        # 모델 출력 결과에서 0 또는 1 추출 (안전장치)
        if '1' in pred:
            y_pred.append(1)
        elif '0' in pred:
            y_pred.append(0)
        else:
            y_pred.append(0) # 판단 불가 시 기본값 정상(0) 처리
            
        if (idx + 1) % 5 == 0:
            print(f" 진행 중... ({idx + 1}/{len(llm_sample)})")
            
    end_time = time.time()
    
    # 실제 정답 데이터 라벨
    y_true = llm_sample['is_attack'].tolist()
    
    # 평가지표 계산
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    
    few_shot_results[p_name] = {
        "Accuracy": acc,
        "F1-Score": f1,
        "Time (s)": round(end_time - start_time, 2)
    }

# 4. 최종 결과 출력 및 가장 정확도가 높은 프롬프트 찾기
print("\n" + "="*60)
print("📊 Few-shot 프롬프트 최적화 비교 결과")
print("="*60)
df_few_shot = pd.DataFrame(few_shot_results).T
print(df_few_shot)

# 가장 정확도(Accuracy)가 높은 프롬프트 자동 선별
best_prompt_name = df_few_shot['Accuracy'].idxmax()
best_accuracy = df_few_shot.loc[best_prompt_name, 'Accuracy']

print("\n" + "-"*60)
print(f"🏆 최적의 프롬프트: {best_prompt_name} (정확도: {best_accuracy*100:.1f}%)")
print("-"*60)


🚀 Few-Shot_기본 (정상1, 공격1) 테스트 시작...
 진행 중... (5/20)
 진행 중... (10/20)
 진행 중... (15/20)
 진행 중... (20/20)

🚀 Few-Shot_강화 (정상2, 공격2 + 설명 형식 지정) 테스트 시작...
 진행 중... (5/20)
 진행 중... (10/20)
 진행 중... (15/20)
 진행 중... (20/20)

📊 Few-shot 프롬프트 최적화 비교 결과
                                   Accuracy  F1-Score  Time (s)
Few-Shot_기본 (정상1, 공격1)                 0.45  0.310345      0.05
Few-Shot_강화 (정상2, 공격2 + 설명 형식 지정)      0.45  0.310345      0.04

------------------------------------------------------------
🏆 최적의 프롬프트: Few-Shot_기본 (정상1, 공격1) (정확도: 45.0%)
------------------------------------------------------------
